<a href="https://colab.research.google.com/github/j22k/Malayalam-TTS/blob/main/Notebooks/preprocess/Preprocess_Dataset_AIkosh_Malayalam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocess Kathbath Malayalam for F5-TTS fine-tuning

**Goal:** turn the Kathbath Malayalam *Test-Known* dataset into the format that F5-TTS fine-tuning expects (`raw.arrow`, `duration.json` and `vocab.txt`). Everything is stored on **Google Drive**, so it survives Colab restarts.

**Pipeline position:** `1. EDA` → **`2. Preprocess (this notebook)`** → `3. Finetune_F5_TTS.ipynb`

### ⚠️ Tested environment: do not change these versions
These exact versions were found by trial and error and work together on Colab. Every text patch in this notebook depends on the **f5-tts 1.1.22** source code.

| Component | Version |
|---|---|
| Colab runtime | **T4 GPU**, Python 3.13.15 |
| torch / torchaudio / torchvision | `2.8.0+cu128` / `2.8.0+cu128` / `0.23.0+cu128` (CUDA 12.8 index) |
| f5-tts | `1.1.22` |
| transformers | `5.16.1` |
| aikosh | `2.0.0` |

### Output (on Google Drive)
```
/content/drive/MyDrive/Datsets/F5-TTS-Malayalam/
├── raw/Kathbath-Malayalam-Test-Known.zip        # original download
├── kathbath/Kathbath-Malayalam-Test-Known/      # extracted WAVs + metadata_finetune.csv
├── scripts/prepare_csv_wavs_malayalam.py        # patched F5-TTS prepare script
└── dataset/
    ├── vocab.txt                                # pretrained Malayalam model vocab (119 chars)
    └── kathbath_malayalam_finetune/             # ← input for the fine-tuning notebook
        ├── raw.arrow
        ├── duration.json
        └── vocab.txt
```
> The Drive folder name `Datsets` (sic) is kept on purpose. Your existing data already lives there.

The notebook is safe to re-run: the download and extraction steps are skipped when their output already exists.

## 1. Install the pinned libraries

**What we do:** install PyTorch 2.8.0 built for CUDA 12.8 first, then `f5-tts==1.1.22` with `transformers==5.16.1`, then the AIKosh client. Install PyTorch **first** so that f5-tts reuses the CUDA 12.8 build.

**What happened:** everything installed without errors.

> If Colab shows a *"Restart session"* prompt, restart, then continue from **step 2** without re-running this cell.

In [ ]:
!pip install -q \
    torch==2.8.0+cu128 \
    torchaudio==2.8.0+cu128 \
    torchvision==0.23.0+cu128 \
    --index-url https://download.pytorch.org/whl/cu128

!pip install -q \
    f5-tts==1.1.22 \
    transformers==5.16.1

!pip install -q aikosh==2.0.0

## 2. Verify the environment

**What we do:** print the installed versions and stop immediately if any of them differs from the tested set.

**What happened:** Python 3.13.15, PyTorch 2.8.0+cu128, Torchaudio 2.8.0+cu128, Torchvision 0.23.0+cu128, Transformers 5.16.1, F5-TTS 1.1.22, CUDA 12.8, GPU Tesla T4.

In [ ]:
import sys
import importlib.metadata

import torch
import torchaudio
import torchvision
import transformers

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchaudio  :", torchaudio.__version__)
print("Torchvision :", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("F5-TTS      :", importlib.metadata.version("f5-tts"))
print("CUDA        :", torch.version.cuda)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

EXPECTED = {
    "torch": "2.8.0+cu128",
    "torchaudio": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128",
    "transformers": "5.16.1",
    "f5-tts": "1.1.22",
}
INSTALLED = {
    "torch": torch.__version__,
    "torchaudio": torchaudio.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "f5-tts": importlib.metadata.version("f5-tts"),
}
mismatch = {k: (INSTALLED[k], v) for k, v in EXPECTED.items() if INSTALLED[k] != v}
assert not mismatch, f"Version mismatch (installed, expected): {mismatch}"
print("\nAll versions match the tested environment.")

## 3. Mount Google Drive and define every path

**What we do:** mount Drive and define **all** paths used by this notebook in one place, then create the folders.

**What happened:** Drive mounted at `/content/drive`, and the project folder `/content/drive/MyDrive/Datsets/F5-TTS-Malayalam` exists.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Datsets/F5-TTS-Malayalam")

RAW_DIR      = PROJECT_DIR / "raw"        # downloaded ZIP
KATHBATH_DIR = PROJECT_DIR / "kathbath"   # extracted dataset
DATASET_DIR  = PROJECT_DIR / "dataset"    # F5-TTS ready data + vocab
SCRIPTS_DIR  = PROJECT_DIR / "scripts"    # patched F5-TTS scripts

for folder in (RAW_DIR, KATHBATH_DIR, DATASET_DIR, SCRIPTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DATASET_ID   = "604b58d2-a04d-4a7b-9c6a-43928b09fc15"
ZIP_NAME     = "Kathbath-Malayalam-Test-Known"
ZIP_PATH     = RAW_DIR / f"{ZIP_NAME}.zip"
DATASET_ROOT = KATHBATH_DIR / ZIP_NAME

MODEL_ID       = "multilingual-tts/F5-TTS-OpenBible-Malayalam"
MODEL_REVISION = "fa4ad6e0d147ac4ab6fdc9e783ed10fbabcde6d3"   # exact snapshot used so far
VOCAB_PATH     = DATASET_DIR / "vocab.txt"

FILTERED_METADATA_PATH = DATASET_ROOT / "metadata_finetune.csv"
SCRIPT_PATH            = SCRIPTS_DIR / "prepare_csv_wavs_malayalam.py"
OUTPUT_DIR             = DATASET_DIR / "kathbath_malayalam_finetune"

print("Project        :", PROJECT_DIR, "| exists:", PROJECT_DIR.exists())
print("ZIP            :", ZIP_PATH)
print("Dataset root   :", DATASET_ROOT)
print("Model vocab    :", VOCAB_PATH)
print("Prepared output:", OUTPUT_DIR)

## 4. Download the dataset from AIKosh (skipped if already on Drive)

**What we do:** read the `AIKOSH_API_KEY` Colab secret and download the dataset ZIP into `raw/`. If the ZIP already exists on Drive, the download is skipped.

**What happened:** `Kathbath-Malayalam-Test-Known.zip` (451 MB) downloaded in about 35 s to `raw/`.

In [ ]:
if ZIP_PATH.exists():
    print("ZIP already on Drive, skipping download:", ZIP_PATH)
else:
    import aikosh
    from google.colab import userdata

    aikosh.set_api_key(userdata.get("AIKOSH_API_KEY"))

    result = aikosh.download(
        {
            "identifier": DATASET_ID,
            "type": "dataset",
            "destination_path": str(RAW_DIR),
        }
    )
    print("Download result:", result)

print("ZIP exists:", ZIP_PATH.exists())
print("ZIP size  :", round(ZIP_PATH.stat().st_size / 1024**3, 2), "GB")

## 5. Extract the ZIP (skipped if already extracted) and verify it

**What we do:** extract into `kathbath/`, then check that `params.json`, `data.json` and all 1767 WAV files are present.

**What happened:** ZIP size 0.44 GB with 1771 entries. After extraction, `params.json`, `data.json` and `audios/` all exist, with **1767 WAV files**.

In [ ]:
import zipfile

if (DATASET_ROOT / "data.json").exists():
    print("Already extracted, skipping:", DATASET_ROOT)
else:
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        print("ZIP entries:", len(z.infolist()))
        z.extractall(KATHBATH_DIR)
    print("Extraction complete.")

print("\nparams.json:", (DATASET_ROOT / "params.json").exists())
print("data.json  :", (DATASET_ROOT / "data.json").exists())
print("audios/    :", (DATASET_ROOT / "audios").is_dir())

wav_count = len(list((DATASET_ROOT / "audios").glob("*.wav")))
print("WAV files  :", wav_count)
assert wav_count == 1767, "Unexpected number of WAV files, so the extraction may be incomplete."

## 6. Download the pretrained model vocabulary

**What we do:** download `vocab.txt` from the Malayalam F5-TTS model, pinned to the same revision as the checkpoint, and keep a copy on Drive. Fine-tuning **must** reuse this exact vocabulary. Its size and order define the model's text-embedding table (119 characters + 1 filler = 120 rows).

**What happened:** vocab size **119**. It starts with `' ', '!', '"', "'", '(', ...` and contains the atomic chillus `ൺ ൻ ർ ൽ ൾ`, ZWNJ/ZWJ and the modern AU sign `ൗ`.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

downloaded_vocab = hf_hub_download(
    repo_id=MODEL_ID,
    filename="vocab.txt",
    revision=MODEL_REVISION,
)
shutil.copy2(downloaded_vocab, VOCAB_PATH)

vocab_list = VOCAB_PATH.read_text(encoding="utf-8").splitlines()
vocab_chars = set(vocab_list)

print("Persistent vocab:", VOCAB_PATH)
print("Vocab size      :", len(vocab_list))
print("First 15 entries:", vocab_list[:15])
print("'ൗ' (U+0D57) in vocab:", "ൗ" in vocab_chars)
assert len(vocab_list) == 119, "Unexpected vocab size, so this is not the vocab the checkpoint was trained with."

## 7. Clean the transcripts and build `metadata_finetune.csv`

**What we do**, for each of the 1767 records:
1. **Exclude speaker `117`** (only 4 clips, see EDA).
2. **Drop clips longer than 20 s** (1 clip). This lets training use a batch size of 2400 frames, since a 20 s clip is 1875 frames.
3. **Normalise the text:** Unicode NFC, then the old AU sign `ൌ` (U+0D4C) → modern `ൗ` (U+0D57), then collapse whitespace. Both spellings sound the same, and only `ൗ` is in the model vocab. **Previously these 101 clips were dropped. Now they are kept.**
4. **Drop anything still out of vocab.** Only the single `ഌ` clip is expected here.
5. Write `audio_file|text` rows with absolute paths, which is the format `prepare_csv_wavs.py` expects.

**Previous run:** the old version dropped 102 clips and kept **1661 clips (4.72 h)**.
**Expected now:** about **1761 clips (about 5.0 h)**: 4 (speaker 117) + 1 (> 20 s) + 1 (`ഌ`) removed.

In [ ]:
import csv
import json
import unicodedata
from collections import Counter

import soundfile as sf

EXCLUDED_SPEAKERS = {"117"}
MAX_DURATION_SEC = 20.0


def normalize_text(text: str) -> str:
    """Normalise a Kathbath transcript to the spelling used by the model vocab."""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("ൌ", "ൗ")   # ൌ (old AU sign) -> ൗ (AU length mark)
    return " ".join(text.split())


with open(DATASET_ROOT / "data.json", "r", encoding="utf-8") as f:
    records = json.load(f)

kept = []
dropped = Counter()
unsupported_chars = Counter()
normalised_count = 0

for record in records:
    if str(record["speaker"]) in EXCLUDED_SPEAKERS:
        dropped["speaker 117"] += 1
        continue

    audio_path = (DATASET_ROOT / record["audioFilename"]).resolve()
    if not audio_path.exists():
        dropped["missing audio"] += 1
        continue

    duration = sf.info(str(audio_path)).duration
    if duration > MAX_DURATION_SEC:
        dropped[f"longer than {MAX_DURATION_SEC:.0f} s"] += 1
        continue

    text = normalize_text(record["text"])
    if text != record["text"]:
        normalised_count += 1

    bad_chars = set(text) - vocab_chars
    if bad_chars:
        dropped["characters not in vocab"] += 1
        unsupported_chars.update(bad_chars)
        continue

    kept.append({"audio_file": str(audio_path), "text": text, "duration": duration})

with open(FILTERED_METADATA_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|", lineterminator="\n")
    writer.writerow(["audio_file", "text"])
    for row in kept:
        writer.writerow([row["audio_file"], row["text"]])

print("Input records     :", len(records))
print("Kept records      :", len(kept))
print("Texts normalised  :", normalised_count)
print("Dropped           :", dict(dropped))
print("Unsupported chars :", {f"{c} (U+{ord(c):04X})": n for c, n in unsupported_chars.items()})
print("Kept duration     :", round(sum(r["duration"] for r in kept) / 3600, 2), "hours")
print("\nWritten:", FILTERED_METADATA_PATH)

## 8. Build the patched F5-TTS preparation script

**What we do:** copy F5-TTS's own `prepare_csv_wavs.py` from the installed 1.1.22 package and apply **two patches**. The result is written to `scripts/prepare_csv_wavs_malayalam.py`.

| Patch | Why |
|---|---|
| `batch_convert_texts` returns the texts unchanged | The stock script converts text to Chinese **pinyin**, which would destroy the Malayalam text |
| `PRETRAINED_VOCAB_PATH` → our Malayalam `vocab.txt` | The stock path points to `data/Emilia_ZH_EN_pinyin/vocab.txt`, which does not exist in a pip install. The first run failed with `AssertionError: pretrained vocab.txt not found` |

The script is regenerated from the installed package every time, so the patches are never applied twice. If an expected code block is not found, the cell stops with an error instead of producing a half-patched script. That would mean the f5-tts version changed.

**What happened:** both patches applied. The pinyin call is gone, and the Malayalam vocab path is present.

In [ ]:
import inspect
import f5_tts.train.datasets.prepare_csv_wavs as prep

source = inspect.getsource(prep)

OLD_PINYIN = '''def batch_convert_texts(texts, polyphone, batch_size=BATCH_SIZE):
    """Convert a list of texts to pinyin in batches."""
    converted_texts = []
    for i in tqdm(
        range(0, len(texts), batch_size),
        total=(len(texts) + batch_size - 1) // batch_size,
        desc="Converting texts to pinyin",
    ):
        batch = texts[i : i + batch_size]
        converted_batch = convert_char_to_pinyin(batch, polyphone=polyphone)
        converted_texts.extend(converted_batch)
    return converted_texts
'''

NEW_PINYIN = '''def batch_convert_texts(texts, polyphone, batch_size=BATCH_SIZE):
    """Keep Malayalam transcripts unchanged; skip pinyin conversion."""
    return texts
'''

OLD_VOCAB = 'PRETRAINED_VOCAB_PATH = files("f5_tts").joinpath("../../data/Emilia_ZH_EN_pinyin/vocab.txt")'
NEW_VOCAB = f'PRETRAINED_VOCAB_PATH = Path(r"{VOCAB_PATH}")'

for name, old, new in [
    ("skip pinyin conversion", OLD_PINYIN, NEW_PINYIN),
    ("Malayalam vocab path", OLD_VOCAB, NEW_VOCAB),
]:
    if source.count(old) != 1:
        raise RuntimeError(f"Patch '{name}': expected code not found. Is f5-tts still 1.1.22?")
    source = source.replace(old, new)
    print("Patched:", name)

SCRIPT_PATH.write_text(source, encoding="utf-8")

print("\nScript                  :", SCRIPT_PATH)
print("Pinyin conversion present:", "convert_char_to_pinyin(batch, polyphone=polyphone)" in source)
print("Malayalam vocab present  :", str(VOCAB_PATH) in source)

## 9. Run the F5-TTS dataset preparation

**What we do:** delete any previous output folder (it only contains generated files), then run the patched script on `metadata_finetune.csv`. In fine-tune mode it:
- reads each WAV and records its duration,
- writes `raw.arrow` (audio path + text) and `duration.json`,
- copies our Malayalam `vocab.txt` into the output folder.

**What happened:** return code **0** (with the 1661-clip metadata from the previous version).

In [ ]:
import shutil
import subprocess
import sys

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    str(FILTERED_METADATA_PATH),
    str(OUTPUT_DIR),
    "--workers", "2",
]

print("Running:", " ".join(cmd), "\n")
result = subprocess.run(cmd, capture_output=True, text=True)

print("========== STDOUT (tail) ==========")
print(result.stdout[-3000:])
print("========== STDERR (tail) ==========")
print(result.stderr[-3000:])
print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Dataset preparation failed, see STDERR above.")

## 10. Verify the prepared dataset

**What we do:** load the output with 🤗 `datasets` and check that:
- `raw.arrow` and `duration.json` have the same number of entries as the metadata CSV,
- the output `vocab.txt` is **byte-identical** to the pretrained model vocab (the token order matters),
- every character in every transcript is in the vocab.

**Previous run:** 1661 samples, 1661 durations, vocab 119, 4.72 h.
**Expected now:** about 1761 samples, about 5.0 h, 0 out-of-vocab characters.

In [ ]:
import json
from datasets import Dataset

raw = Dataset.from_file(str(OUTPUT_DIR / "raw.arrow"))

with open(OUTPUT_DIR / "duration.json", "r", encoding="utf-8") as f:
    durations = json.load(f)["duration"]

out_vocab_text = (OUTPUT_DIR / "vocab.txt").read_text(encoding="utf-8")
all_text_chars = set("".join(raw["text"]))

print("raw.arrow samples :", len(raw))
print("duration entries  :", len(durations))
print("metadata rows     :", len(kept))
print("vocab size        :", len(out_vocab_text.splitlines()))
print("total duration    :", round(sum(durations) / 3600, 2), "hours")
print("min / max duration:", round(min(durations), 2), "/", round(max(durations), 2), "s")

print("\nFirst sample:")
print("  audio_path:", raw[0]["audio_path"])
print("  text      :", raw[0]["text"])
print("  duration  :", raw[0]["duration"])

assert len(raw) == len(durations) == len(kept), "Sample count mismatch"
assert out_vocab_text == VOCAB_PATH.read_text(encoding="utf-8"), "Output vocab differs from the model vocab"
assert not (all_text_chars - vocab_chars), f"Out-of-vocab chars: {all_text_chars - vocab_chars}"
print("\nDataset is ready for fine-tuning:", OUTPUT_DIR)

## ✅ Done

The prepared dataset is at `dataset/kathbath_malayalam_finetune/` on Drive.

**Next:** open `Notebooks/Finetune_F5_TTS.ipynb`. It reads this folder and `dataset/vocab.txt` directly.